# MP3 to YouTube Workflow

This notebook is organized into two big stages:

1. Prepare inputs, convert audio to subtitle-burned videos, and persist outputs to Google Drive.
2. Upload the rendered videos from Google Drive to YouTube.

## Stage 0 - Install Dependencies

Install FFmpeg and the Python packages used by both rendering and upload steps.

In [ ]:
# Stage 0 - Install dependencies
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install --upgrade --quiet \
    google-auth \
    google-auth-oauthlib \
    google-auth-httplib2 \
    google-api-python-client \
    pillow

## Shared Helpers

Reusable functions for image preparation, FFmpeg rendering, manifest generation, and YouTube API operations.

In [ ]:
# Shared helpers used by both big stages
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from datetime import datetime
from pathlib import Path

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from PIL import Image


SRT_TIMESTAMP_RE = re.compile(r'^(\d{2}):(\d{2}):(\d{2}),(\d{3})$')


def run_ffmpeg(cmd):
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"FFmpeg failed with exit code {exc.returncode}") from exc


def optimize_cover_image(image_path: Path, local_cover: Path, size=(1280, 720)) -> Path:
    image_path = Path(image_path)
    local_cover = Path(local_cover)
    local_cover.parent.mkdir(parents=True, exist_ok=True)

    if not image_path.exists():
        print('Cover image missing. Creating a black placeholder cover.')
        run_ffmpeg([
            'ffmpeg', '-y',
            '-f', 'lavfi', '-i', f'color=c=black:s={size[0]}x{size[1]}:d=1',
            '-frames:v', '1', str(image_path),
        ])

    with Image.open(image_path) as img:
        img = img.convert('RGB')
        img.thumbnail(size, Image.Resampling.LANCZOS)
        canvas = Image.new('RGB', size, color='black')
        offset = ((size[0] - img.width) // 2, (size[1] - img.height) // 2)
        canvas.paste(img, offset)
        canvas.save(local_cover, format='PNG')

    print(f'Optimized cover cached at: {local_cover}')
    return local_cover


def parse_srt_timestamp(value: str) -> int:
    match = SRT_TIMESTAMP_RE.match(value.strip())
    if not match:
        raise ValueError(f'Invalid SRT timestamp: {value}')
    hours, minutes, seconds, milliseconds = map(int, match.groups())
    return ((hours * 60 + minutes) * 60 + seconds) * 1000 + milliseconds


def format_srt_timestamp(total_ms: int) -> str:
    total_ms = max(0, total_ms)
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    seconds, milliseconds = divmod(remainder, 1_000)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}'


def shift_srt_file(source_path: Path, destination_path: Path, shift_seconds: float) -> Path:
    shift_ms = int(round(shift_seconds * 1000))
    shifted_lines = []
    for raw_line in Path(source_path).read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if '-->' not in line:
            shifted_lines.append(raw_line)
            continue
        start_text, end_text = [part.strip() for part in line.split('-->', 1)]
        shifted_start = format_srt_timestamp(parse_srt_timestamp(start_text) + shift_ms)
        shifted_end = format_srt_timestamp(parse_srt_timestamp(end_text) + shift_ms)
        shifted_lines.append(f'{shifted_start} --> {shifted_end}')
    Path(destination_path).write_text('\n'.join(shifted_lines) + '\n', encoding='utf-8')
    return destination_path


def build_subtitle_filter(local_srt: Path) -> str:
    subtitle_path = str(local_srt).replace('\\', '/')
    subtitle_path = subtitle_path.replace(':', r'\:').replace("'", r"\'")
    style = (
        "FontName=Arial,FontSize=20,PrimaryColour=&HFFFFFF&,"
        "OutlineColour=&H000000&,BorderStyle=1,Outline=2,Shadow=1,"
        "MarginV=28,Alignment=2"
    )
    return (
        'scale=1280:720:force_original_aspect_ratio=decrease,'
        'pad=1280:720:(ow-iw)/2:(oh-ih)/2,'
        f"subtitles='{subtitle_path}':force_style='{style}'"
    )


def convert_audio_to_video(audio_path: Path, image_path: Path, output_path: Path, subtitle_path: Path | None = None,
                           video_preset='ultrafast', crf='30', fps='2', audio_bitrate='128k',
                           opening_silence_seconds=0.0, ending_silence_seconds=0.0,
                           subtitle_offset_seconds=0.0) -> Path:
    audio_path = Path(audio_path)
    image_path = Path(image_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if not audio_path.exists():
        raise FileNotFoundError(f'Audio missing: {audio_path}')
    if not image_path.exists():
        raise FileNotFoundError(f'Image missing: {image_path}')
    if subtitle_path is not None and not Path(subtitle_path).exists():
        raise FileNotFoundError(f'Subtitle missing: {subtitle_path}')

    temp_dir = Path(tempfile.mkdtemp(prefix='yt_render_'))
    local_audio = temp_dir / audio_path.name
    local_image = temp_dir / image_path.name
    local_output = temp_dir / output_path.name
    local_srt = temp_dir / subtitle_path.name if subtitle_path else None

    shutil.copy2(audio_path, local_audio)
    shutil.copy2(image_path, local_image)
    if subtitle_path:
        if subtitle_offset_seconds != 0:
            shift_srt_file(subtitle_path, local_srt, subtitle_offset_seconds)
        else:
            shutil.copy2(subtitle_path, local_srt)

    vf = build_subtitle_filter(local_srt) if local_srt else 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2'
    audio_filters = []
    if opening_silence_seconds > 0:
        delay_ms = int(round(opening_silence_seconds * 1000))
        audio_filters.append(f'adelay={delay_ms}:all=1')
    if ending_silence_seconds > 0:
        audio_filters.append(f'apad=pad_dur={ending_silence_seconds}')

    cmd = [
        'ffmpeg', '-y',
        '-hide_banner', '-loglevel', 'error', '-stats',
        '-loop', '1', '-i', str(local_image),
        '-i', str(local_audio),
        '-vf', vf,
        '-r', str(fps),
        '-c:v', 'libx264', '-tune', 'stillimage', '-preset', str(video_preset), '-crf', str(crf),
        '-pix_fmt', 'yuv420p',
        *( ['-af', ','.join(audio_filters)] if audio_filters else [] ),
        '-c:a', 'aac', '-b:a', str(audio_bitrate),
        '-shortest', '-movflags', '+faststart',
        '-map', '0:v:0', '-map', '1:a:0',
        str(local_output),
    ]

    print(f'Rendering {audio_path.name} -> {output_path.name}')
    start_time = time.time()
    try:
        run_ffmpeg(cmd)
        shutil.move(str(local_output), str(output_path))
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

    print(f'Saved {output_path.name} in {time.time() - start_time:.2f}s')
    return output_path


def upload_to_youtube(youtube, video_path: Path, title: str, description: str, tags=None,
                      privacy_status='public', made_for_kids=False, category_id='22') -> str:
    tags = tags or []
    body = {
        'snippet': {
            'title': title,
            'description': description,
            'categoryId': category_id,
            'tags': tags,
        },
        'status': {
            'privacyStatus': privacy_status,
            'selfDeclaredMadeForKids': bool(made_for_kids),
        },
    }
    media = MediaFileUpload(str(video_path), chunksize=-1, resumable=True, mimetype='video/mp4')
    request = youtube.videos().insert(part='snippet,status', body=body, media_body=media)
    response = None
    retry = 0
    while response is None:
        try:
            _, response = request.next_chunk()
        except Exception:
            retry += 1
            if retry > 5:
                raise
            time.sleep(2 * retry)
    return response['id']


def upload_caption(youtube, video_id: str, caption_path: Path, name='German', language='de'):
    body = {
        'snippet': {
            'videoId': video_id,
            'language': language,
            'name': name,
            'isDraft': False,
        }
    }
    media = MediaFileUpload(str(caption_path), mimetype='application/octet-stream', resumable=True)
    request = youtube.captions().insert(part='snippet', body=body, media_body=media)
    response = None
    while response is None:
        _, response = request.next_chunk()
    print(f'Caption uploaded for video {video_id}')
    return response


def get_playlist_id_by_title(youtube, playlist_title: str):
    request = youtube.playlists().list(part='snippet', mine=True, maxResults=50)
    while request:
        response = request.execute()
        for playlist in response.get('items', []):
            if playlist['snippet']['title'] == playlist_title:
                return playlist['id']
        request = youtube.playlists().list_next(request, response)
    raise ValueError(f"Playlist named '{playlist_title}' not found")


def add_to_playlist(youtube, video_id: str, playlist_id: str):
    youtube.playlistItems().insert(
        part='snippet',
        body={'snippet': {'playlistId': playlist_id, 'resourceId': {'kind': 'youtube#video', 'videoId': video_id}}}
    ).execute()
    print(f'Added video {video_id} to playlist {playlist_id}')


def extract_plain_transcript_from_srt(srt_path: Path) -> str:
    text = Path(srt_path).read_text(encoding='utf-8')
    lines = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.isdigit():
            continue
        if '-->' in line:
            continue
        lines.append(line)
    return '\n'.join(lines)


def build_manifest(mp3_folder: Path, output_folder: Path, burn_subtitles=True, output_name_prefix=''):
    items = []
    for mp3 in sorted(Path(mp3_folder).glob('*.mp3')):
        srt = Path(mp3_folder) / f'{mp3.stem}.srt'
        subtitle_source_path = srt if srt.exists() else None
        prefixed_stem = f'{output_name_prefix}-{mp3.stem}' if output_name_prefix else mp3.stem
        items.append({
            'stem': mp3.stem,
            'output_stem': prefixed_stem,
            'audio_path': str(mp3),
            'subtitle_path': str(srt) if burn_subtitles and srt.exists() else None,
            'subtitle_source_path': str(subtitle_source_path) if subtitle_source_path else None,
            'video_path': str(Path(output_folder) / f'{prefixed_stem}.mp4'),
            'title': prefixed_stem,
            'description': f'Auto-uploaded version of "{mp3.stem}".',
        })
    return items


def clear_folder_contents(folder_path: Path):
    folder_path = Path(folder_path)
    folder_path.mkdir(parents=True, exist_ok=True)
    for path in folder_path.glob('*'):
        if path.is_file():
            path.unlink()
        elif path.is_dir():
            shutil.rmtree(path)


def find_existing_video_for_stem(output_folder: Path, stem: str) -> Path | None:
    output_folder = Path(output_folder)
    exact_match = output_folder / f'{stem}.mp4'
    if exact_match.exists():
        return exact_match

    candidates = sorted(
        output_folder.glob(f'*-{stem}.mp4'),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None




def prepare_caption_upload_file(source_srt: Path, subtitle_offset_seconds: float) -> Path:
    if subtitle_offset_seconds == 0:
        return Path(source_srt)

    temp_dir = Path(tempfile.mkdtemp(prefix='yt_caption_'))
    shifted_path = temp_dir / Path(source_srt).name
    shift_srt_file(Path(source_srt), shifted_path, subtitle_offset_seconds)
    return shifted_path

def build_upload_manifest_from_existing_outputs(input_folder: Path, output_folder: Path):
    items = []
    for mp3 in sorted(Path(input_folder).glob('*.mp3')):
        video_path = find_existing_video_for_stem(output_folder, mp3.stem)
        srt = Path(input_folder) / f'{mp3.stem}.srt'
        subtitle_source_path = srt if srt.exists() else None
        items.append({
            'stem': mp3.stem,
            'output_stem': video_path.stem if video_path else mp3.stem,
            'audio_path': str(mp3),
            'subtitle_path': str(srt) if subtitle_source_path else None,
            'subtitle_source_path': str(subtitle_source_path) if subtitle_source_path else None,
            'video_path': str(video_path) if video_path else str(Path(output_folder) / f'{mp3.stem}.mp4'),
            'title': video_path.stem if video_path else mp3.stem,
            'description': f'Auto-uploaded version of "{mp3.stem}".',
        })
    return items


# Stage 1 - Prepare and Render to Google Drive

This stage authenticates Google Drive access, gathers inputs, renders videos, and saves reports for later publishing.

## Stage 1A - Google Drive Auth and Workspace Setup

Mount Google Drive and define the folders and render settings used by the conversion pipeline.

In [ ]:
# Stage 1A - Google Drive auth and workspace setup
from google.colab import drive
from google.colab import files

DRIVE_ROOT = Path('/content/drive/MyDrive/GermanListeningYoutube')
INPUT_FOLDER = DRIVE_ROOT / 'inputs'
OUTPUT_FOLDER = DRIVE_ROOT / 'rendered_videos'
REPORT_FOLDER = DRIVE_ROOT / 'reports'
IMAGE_FILE = DRIVE_ROOT / 'cover.png'
LOCAL_COVER = Path('/tmp/optimized_cover.png')
MANIFEST_FILE = REPORT_FOLDER / 'render_manifest.json'
RENDER_REPORT_FILE = REPORT_FOLDER / 'render_results.json'

BURN_SUBTITLES = True
FORCE_REBUILD = False
VIDEO_PRESET = 'ultrafast'
VIDEO_CRF = '30'
VIDEO_FPS = '2'
AUDIO_BITRATE = '128k'
OPENING_SILENCE_SECONDS = 0.5
ENDING_SILENCE_SECONDS = 1.0
SUBTITLE_OFFSET_SECONDS = 0.5
OUTPUT_NAME_PREFIX = datetime.now().strftime('%y-%m-%d-%H')


drive.mount('/content/drive', force_remount=False)
for folder in [INPUT_FOLDER, OUTPUT_FOLDER, REPORT_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

print('Workspace ready:')
print(f'  Inputs : {INPUT_FOLDER}')
print(f'  Outputs: {OUTPUT_FOLDER}')
print(f'  Reports: {REPORT_FOLDER}')
print(f'  Cover  : {IMAGE_FILE}')
print(f'  Prefix : {OUTPUT_NAME_PREFIX}')
print(f'  Silence: start={OPENING_SILENCE_SECONDS}s end={ENDING_SILENCE_SECONDS}s')
print(f'  Subtitle offset: {SUBTITLE_OFFSET_SECONDS}s')


## Stage 1B - Upload Inputs or Inspect Existing Files

Use this when you want to upload fresh `.mp3` and matching `.srt` files into the Drive-backed input folder.

In [ ]:
# Stage 1B - Upload inputs or inspect what is already in Drive
# Optional housekeeping before upload or conversion.
CLEAR_INPUT_FOLDER = False
CLEAR_OUTPUT_FOLDER = False
UPLOAD_NEW_INPUTS = False

if CLEAR_INPUT_FOLDER:
    clear_folder_contents(INPUT_FOLDER)
    print('Input folder cleared.')

if CLEAR_OUTPUT_FOLDER:
    clear_folder_contents(OUTPUT_FOLDER)
    print('Output folder cleared.')

if UPLOAD_NEW_INPUTS:
    uploaded = files.upload()
    for filename, data in uploaded.items():
        target = INPUT_FOLDER / filename
        with open(target, 'wb') as handle:
            handle.write(data)
    print(f'Uploaded {len(uploaded)} file(s) to {INPUT_FOLDER}')
else:
    print('Skipping interactive upload. Using files already in Drive.')

current_input_files = sorted(INPUT_FOLDER.glob('*'))
current_output_files = sorted(OUTPUT_FOLDER.glob('*'))
print(f'Found {len(current_input_files)} input file(s):')
for path in current_input_files:
    print(f'  - {path.name}')

print(f'Found {len(current_output_files)} output file(s):')
for path in current_output_files[:10]:
    print(f'  - {path.name}')
if len(current_output_files) > 10:
    print(f'  ... and {len(current_output_files) - 10} more')


## Stage 1C - Validate Inputs and Build Manifest

Pair each audio file with its `.srt` subtitle file, then save a manifest for rendering and later upload.

In [ ]:
# Stage 1C - Validate inputs and build the render manifest
if not IMAGE_FILE.exists():
    print('Cover file not found yet. A placeholder will be created during rendering.')

manifest = build_manifest(
    INPUT_FOLDER,
    OUTPUT_FOLDER,
    burn_subtitles=BURN_SUBTITLES,
    output_name_prefix=OUTPUT_NAME_PREFIX,
)
if not manifest:
    raise RuntimeError(f'No MP3 files found in {INPUT_FOLDER}')

missing_srt = [item['stem'] for item in manifest if BURN_SUBTITLES and not item['subtitle_path']]
print(f'Manifest contains {len(manifest)} video job(s).')
print('Output names preview:')
for item in manifest[:5]:
    print(f"  - {item['output_stem']}.mp4")
if len(manifest) > 5:
    print(f'  ... and {len(manifest) - 5} more')
if missing_srt:
    print('These files are missing .srt subtitles and will render without burned subtitles:')
    for stem in missing_srt:
        print(f'  - {stem}')

REPORT_FOLDER.mkdir(parents=True, exist_ok=True)
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Manifest saved to {MANIFEST_FILE}')


## Stage 1D - Convert Inputs into Videos

Render MP3 files into MP4 videos, optionally burning subtitles directly into the video.

In [ ]:
# Stage 1D - Convert inputs into Drive-persisted videos
optimized_cover = optimize_cover_image(IMAGE_FILE, LOCAL_COVER)
render_results = []

for item in manifest:
    audio_path = Path(item['audio_path'])
    video_path = Path(item['video_path'])
    subtitle_path = Path(item['subtitle_path']) if item['subtitle_path'] else None

    if video_path.exists() and not FORCE_REBUILD:
        print(f'Skipping existing video: {video_path.name}')
        render_results.append({'stem': item['stem'], 'status': 'skipped', 'video_path': str(video_path)})
        continue

    try:
        convert_audio_to_video(
            audio_path=audio_path,
            image_path=optimized_cover,
            output_path=video_path,
            subtitle_path=subtitle_path,
            video_preset=VIDEO_PRESET,
            crf=VIDEO_CRF,
            fps=VIDEO_FPS,
            audio_bitrate=AUDIO_BITRATE,
            opening_silence_seconds=OPENING_SILENCE_SECONDS,
            ending_silence_seconds=ENDING_SILENCE_SECONDS,
            subtitle_offset_seconds=SUBTITLE_OFFSET_SECONDS,
        )
        render_results.append({'stem': item['stem'], 'status': 'rendered', 'video_path': str(video_path)})
    except Exception as exc:
        print(f'Failed to render {audio_path.name}: {exc}')
        render_results.append({'stem': item['stem'], 'status': 'failed', 'error': str(exc)})

RENDER_REPORT_FILE.write_text(json.dumps(render_results, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Render report saved to {RENDER_REPORT_FILE}')


## Stage 1E - Review Rendered Outputs

Check which videos were rendered successfully before moving on to YouTube publishing.

In [ ]:
# Stage 1E - Review Stage 1 outputs before publishing
rendered = [item for item in render_results if item['status'] in {'rendered', 'skipped'}]
failed = [item for item in render_results if item['status'] == 'failed']

print(f'Render-ready videos: {len(rendered)}')
for item in rendered:
    print(f"  - {Path(item['video_path']).name} ({item['status']})")

if failed:
    print(f'Failed renders: {len(failed)}')
    for item in failed:
        print(f"  - {item['stem']}: {item['error']}")
else:
    print('No render failures.')

# Stage 2 - Publish Rendered Videos to YouTube

This stage authenticates the YouTube account and uploads the already-rendered videos from Google Drive.

## Stage 2A - YouTube Upload Configuration

Set playlist, privacy, tags, and credential locations for the publishing step.

In [ ]:
# Stage 2A - YouTube auth configuration
PLAYLIST_NAME = 'DAF_B1'
YOUTUBE_PRIVACY_STATUS = 'public'
UPLOAD_CAPTION_TRACK = True
YOUTUBE_TAGS = ['podcast', 'auto-upload']
CATEGORY_ID = '22'
REUSE_EXISTING_OUTPUTS_FOR_UPLOAD = True
ALLOW_OUTPUT_SCAN_FALLBACK = False
USE_SAVED_YOUTUBE_TOKEN = True

YT_CREDENTIALS_DIR = DRIVE_ROOT / 'yt_credentials'
CLIENT_SECRET_FILE = YT_CREDENTIALS_DIR / 'client_secret.json'
TOKEN_FILE = YT_CREDENTIALS_DIR / 'token.json'
YOUTUBE_REPORT_FILE = REPORT_FOLDER / 'youtube_upload_results.json'

print(f'YouTube credentials dir: {YT_CREDENTIALS_DIR}')
print(f'Expected client secret : {CLIENT_SECRET_FILE}')
print(f'Reuse existing outputs : {REUSE_EXISTING_OUTPUTS_FOR_UPLOAD}')
print(f'Allow output scan fallback: {ALLOW_OUTPUT_SCAN_FALLBACK}')
print(f'Use saved YouTube token : {USE_SAVED_YOUTUBE_TOKEN}')


## Stage 2B - Authenticate the YouTube Account

Run OAuth and build the YouTube API client used for uploads and playlist management.

In [ ]:
# Stage 2B - Authenticate the YouTube account
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'
YT_CREDENTIALS_DIR.mkdir(parents=True, exist_ok=True)

if not CLIENT_SECRET_FILE.exists():
    raise FileNotFoundError(f'Missing OAuth client secret: {CLIENT_SECRET_FILE}')

client_config = json.loads(CLIENT_SECRET_FILE.read_text(encoding='utf-8'))
if 'installed' not in client_config:
    raise ValueError("client_secret.json must be a Desktop OAuth client with top-level key 'installed'.")

redirect_uris = set(client_config['installed'].get('redirect_uris', [])) | {'http://localhost'}
client_config['installed']['redirect_uris'] = list(redirect_uris)

scopes = [
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube',
    'https://www.googleapis.com/auth/youtube.force-ssl',
]

creds = None
if USE_SAVED_YOUTUBE_TOKEN and TOKEN_FILE.exists():
    creds = Credentials.from_authorized_user_info(json.loads(TOKEN_FILE.read_text(encoding='utf-8')), scopes=scopes)
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
        TOKEN_FILE.write_text(creds.to_json(), encoding='utf-8')
    if creds and creds.valid:
        print('Reusing saved YouTube token.')

if not creds or not creds.valid:
    flow = InstalledAppFlow.from_client_config(client_config, scopes)
    flow.redirect_uri = 'http://localhost'
    auth_url, _ = flow.authorization_url(access_type='offline', include_granted_scopes='true', prompt='consent')

    print('Open this URL and grant all requested YouTube permissions:')
    print(auth_url)
    redirected_url = input('Paste the full redirected http://localhost URL here:\n> ').strip()

    flow.fetch_token(authorization_response=redirected_url)
    creds = flow.credentials
    TOKEN_FILE.write_text(creds.to_json(), encoding='utf-8')
    print('Saved refreshed YouTube token.')
youtube = build('youtube', 'v3', credentials=creds)

sample = youtube.playlists().list(part='snippet', mine=True, maxResults=3).execute()
print('Authenticated. Sample playlists:', [it['snippet']['title'] for it in sample.get('items', [])])

## Stage 2C - Upload Rendered Videos

Upload the MP4 files from Google Drive to YouTube and optionally attach playlist entries and caption tracks.

In [ ]:
# Stage 2C - Upload rendered videos from Drive to YouTube
if REUSE_EXISTING_OUTPUTS_FOR_UPLOAD:
    if MANIFEST_FILE.exists():
        manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
        print('Reusing rendered outputs via the saved render manifest.')
    elif ALLOW_OUTPUT_SCAN_FALLBACK:
        manifest = build_upload_manifest_from_existing_outputs(INPUT_FOLDER, OUTPUT_FOLDER)
        print('Manifest missing, so falling back to scanning existing MP4 files in the output folder.')
    else:
        raise FileNotFoundError(
            f'Render manifest missing: {MANIFEST_FILE}. Run Stage 1C first or set ALLOW_OUTPUT_SCAN_FALLBACK = True.'
        )
else:
    if not MANIFEST_FILE.exists():
        raise FileNotFoundError(f'Render manifest missing: {MANIFEST_FILE}')
    manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
    print('Using render manifest for upload.')

playlist_id = None
try:
    playlist_id = get_playlist_id_by_title(youtube, PLAYLIST_NAME)
except Exception as exc:
    print(f'Playlist lookup failed: {exc}. Videos will upload without playlist assignment.')

upload_results = []
for item in manifest:
    video_path = Path(item['video_path'])
    subtitle_source_path = Path(item['subtitle_source_path']) if item['subtitle_source_path'] else None
    caption_upload_path = None

    if not video_path.exists():
        print(f'Skipping missing rendered video: {video_path.name}')
        upload_results.append({'stem': item['stem'], 'status': 'missing_video'})
        continue

    description = item['description']
    if subtitle_source_path and subtitle_source_path.exists():
        description += '\n\nTRANSCRIPT\n' + extract_plain_transcript_from_srt(subtitle_source_path)

    try:
        print(f'Uploading {video_path.name} to YouTube...')
        video_id = upload_to_youtube(
            youtube,
            video_path=video_path,
            title=item['title'],
            description=description,
            tags=YOUTUBE_TAGS,
            privacy_status=YOUTUBE_PRIVACY_STATUS,
            category_id=CATEGORY_ID,
        )
        result = {'stem': item['stem'], 'title': item['title'], 'status': 'uploaded', 'video_id': video_id, 'video_path': str(video_path)}

        if playlist_id:
            try:
                add_to_playlist(youtube, video_id, playlist_id)
                result['playlist_id'] = playlist_id
            except Exception as exc:
                result['playlist_error'] = str(exc)
                print(f'Playlist add failed for {item["stem"]}: {exc}')

        if UPLOAD_CAPTION_TRACK and subtitle_source_path and subtitle_source_path.exists():
            try:
                caption_upload_path = prepare_caption_upload_file(subtitle_source_path, SUBTITLE_OFFSET_SECONDS)
                upload_caption(youtube, video_id, caption_upload_path)
            except Exception as exc:
                result['caption_error'] = str(exc)
                print(f'Caption upload failed for {item["stem"]}: {exc}')
            finally:
                if caption_upload_path and caption_upload_path != subtitle_source_path:
                    shutil.rmtree(caption_upload_path.parent, ignore_errors=True)

        upload_results.append(result)
    except Exception as exc:
        print(f'Upload failed for {item["stem"]}: {exc}')
        upload_results.append({'stem': item['stem'], 'title': item['title'], 'status': 'failed', 'error': str(exc), 'video_path': str(video_path)})

YOUTUBE_REPORT_FILE.write_text(json.dumps(upload_results, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'YouTube upload report saved to {YOUTUBE_REPORT_FILE}')


## Stage 2D - Final Summary

Review render and upload results from the reports saved in Google Drive.

In [ ]:
# Stage 2D - Summary
render_report = json.loads(RENDER_REPORT_FILE.read_text(encoding='utf-8')) if RENDER_REPORT_FILE.exists() else []
upload_report = json.loads(YOUTUBE_REPORT_FILE.read_text(encoding='utf-8')) if YOUTUBE_REPORT_FILE.exists() else []

print('Render summary:')
for item in render_report:
    status = item['status']
    name = item.get('video_path', item['stem'])
    print(f'  - {Path(name).name if str(name).endswith(".mp4") else name}: {status}')

print('\nUpload summary:')
for item in upload_report:
    suffix = f" (video_id={item['video_id']})" if item.get('video_id') else ''
    label = item.get('title') or (Path(item['video_path']).stem if item.get('video_path') else item['stem'])
    print(f'  - {label}: {item["status"]}{suffix}')
